# Orbit Wars — Colab Training Pipeline (v2 OrbitNet ExIt, producer-anchored)

GPU-accelerated training on Colab with Google Drive persistence.

**Re-anchored 2026-06-11** (`rl_research/LEADERBOARD_CLIMB_PLAN.md` Phase 2.2): the
BC teacher, the ExIt collection opponent, and the eval gate are all the **producer
flow-diff tier** (LB ~1242, our shipped base `agents/v5/`). The apex-era pipelines
(ppo / v3 / v4 / exit_blend / exit_neural / exit_rollout) were retired — post-mortems
in `rl_research/EXPLORED_AND_ABANDONED.md`.

Pipelines (set `PIPELINE` in the config cell):
1. **`producer256_v3`** (default) — THE live run (Phase 2.2c): the champion recipe
   (fresh BC from producer-mirror demos, producer opponent, Gumbel ON) plus ONE
   variable — `exit.arrival_horizon`, the **arrival-resolving search horizon**.
   Fixes the root cause of the iter-25 plateau: at fixed depth-12 only ~17% of
   candidates ARRIVE inside the lookahead, unresolved leaves eval identical to
   hold, and pi' degenerates to the prior on ~5/6 decisions (floss≈ln4).
2. **`producer256_v2`** — Phase 2.2b data-maxx (mixed pool + 3× data). GATE
   FAILED 2026-06-11 (mildly regressed the champion); kept for reference.
3. **`producer256`** — the first Phase 2.2 run: fresh BC from **producer-MIRROR
   demos** (seats side-alternated) → Gumbel ExIt vs producer, embed-256.
   Produced the current champion (`..._a100/ckpt_000025.pt`).
4. **`embed256`** — the 2026-06-10 capacity-A/B arm B, kept for comparison
   (BC demos vs random, P0-only collection).
5. **`exit`** — embed-128 baseline on the same producer anchoring.

### A100 efficiency note
The OrbitNet is tiny (~2M params at embed-256), so the **GPU is never the compute
bottleneck** — game collection and search are CPU-bound, and producer-tier opponents
are ~50-100× slower per step than the retired apex. We therefore:
- fan BC demo collection, ExIt collection, search, and eval across **all vCPUs**,
- enable **TF32** + large batches so the A100 chews through BC / distillation,
- **cache producer demos on Drive** so the expensive collection runs once
  (seed it from a locally collected cache via rclone to skip it entirely).

**Gate discipline:** in-notebook eval is a coarse signal. The shipping gate is the
local arena (`scripts/arena.py`, real Kaggle env, side-alternated paired seeds,
n≥120 mirror games) vs the public pool AND vs producer/v5. For `producer256_v3`
the bar is the **iter-25 champion**: beat it head-to-head AND raise the
4-opponent pool mean (>45%), n≥30/pair.

In [ ]:
#@title 1. Setup: Mount Drive, Clone Repo, Install Dependencies

import os, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/orbit_wars_outputs'
for sub in ['', '/checkpoints', '/logs', '/submissions']:
    os.makedirs(DRIVE_OUTPUT + sub, exist_ok=True)
print(f'Drive output dir: {DRIVE_OUTPUT}')

REPO_URL = 'https://github.com/oshbocker/orbit_wars.git'  # <-- EDIT if needed
REPO_DIR = '/content/orbit_wars'
if os.path.exists(REPO_DIR):
    print('Repo present, pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working dir: {os.getcwd()}')

# v2 needs only kaggle-environments + the usual scientific stack (no SB3).
os.system('pip install -q --upgrade "kaggle-environments>=1.28.0" pyyaml tensorboard')
print('\nSetup complete.')

In [ ]:
#@title 2. GPU Verification + A100 performance flags

import os, torch

print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print(f'GPU: {name}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    # A100 / Ampere: enable TF32 for fast fp32 matmul (free speedup).
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    DEVICE = 'cuda'
else:
    print('WARNING: no GPU. Runtime > Change runtime type > GPU (A100).')
    DEVICE = 'cpu'

CPU_COUNT = os.cpu_count() or 8
print(f'\nDevice: {DEVICE} | vCPUs: {CPU_COUNT}')
print('Note: collection/search are CPU-bound; the GPU accelerates neural training.')

# Silence kaggle_environments' noisy OpenSpiel INFO logs (it sets these two
# loggers to INFO + attaches a stdout handler at import; .disabled survives that
# because the module never resets it). Run once per kernel, before the eval/gate
# cells import kaggle_environments. Surgical: only these loggers, other INFO kept.
import logging
for _ln in ('kaggle_environments.envs.open_spiel_env.open_spiel_env',
            'kaggle_environments.core_harness'):
    logging.getLogger(_ln).disabled = True

## Configuration

Pick the pipeline and apply A100-tuned overrides. The base YAMLs
(`configs/v2_exit_producer256_v3.yaml`, `configs/v2_exit_producer256_v2.yaml`,
`configs/v2_exit_producer256.yaml`, `configs/v2_exit_embed256.yaml`,
`configs/v2_exit.yaml`) are all producer-anchored (Gumbel search ON); here we
scale batch sizes for the GPU, fan collection/search/eval across the vCPUs, and
point all outputs (incl. the cached producer demo set) at Google Drive.

**`producer256_v3`** is the Phase 2.2c arrival-horizon run: byte-identical to the
champion recipe (`producer256`) except `exit.arrival_horizon: true` (+margin/cap),
which sims each decision deep enough that every candidate's fleet ARRIVES before
the leaf is scored. Mechanism probe (local, 2026-06-11): hostile-candidate
resolution 0%→100% @p50, capture-capable decisions with live q-spread 35%→65%,
fraction-entropy p10 → ~0.02. Watch the iter log for **floss dropping below ln4
(≈1.386)** — that's the sizing signal distilling, the thing every prior run
lacked. Search cost is ~3-5× sims/candidate; search_workers=vCPUs keeps it off
the critical path (collect dominates). Known risk (watch, don't pre-fix): the
passive in-sim opponent never reinforces → longer horizons may overrate distant
captures; if the gate regresses with an aggression signature, the lever is
`arrival_horizon_cap`/`arrival_settle_margin`.

**After training:** download the checkpoint locally
(`uv run python scripts/download_checkpoint.py --run <run_name> --all-ckpts`)
and gate with `scripts/arena.py`: beat
`v2_exit_producer256_a100/ckpt_000025.pt` head-to-head AND raise the 4-opponent
pool mean (>45%), n>=30/pair; mirror vs producer/v5 at n>=60. In-notebook eval
numbers are coarse signals only.

In [ ]:
#@title 3. Experiment Config

from v2.config import load_v2_config

# Post-re-anchor (2026-06-11): everything optimizes vs the PRODUCER tier
# (rl_research/LEADERBOARD_CLIMB_PLAN.md Phase 2.2).
#   producer256_v3 (default) -- Phase 2.2c: the champion recipe + ONE variable,
#       exit.arrival_horizon (arrival-resolving search depth: min(cap, max
#       candidate travel time + settle margin), uniform per decision incl.
#       hold). Fixes the horizon-blind search (only ~17% of candidates arrived
#       inside depth-12 -> unresolved leaves eval == hold -> pi' = prior on
#       ~5/6 decisions, floss stuck at ln4, iter-25 plateau).
#       configs/v2_exit_producer256_v3.yaml
#   producer256_v2 -- Phase 2.2b data-maxx (mixed pool, 3x data, warm-start).
#       GATE FAILED 2026-06-11 -- kept for reference.
#       configs/v2_exit_producer256_v2.yaml
#   producer256 -- the first re-anchor run (fresh BC from producer-MIRROR demos,
#       side-alternated seats). Produced the champion ckpt_000025.
#       configs/v2_exit_producer256.yaml
#   embed256 -- 2026-06-10 capacity-A/B arm B kept for comparison (demos vs
#       random, P0-only collection). configs/v2_exit_embed256.yaml
#   exit -- embed-128 baseline. configs/v2_exit.yaml
PIPELINE = 'producer256_v3'  #@param ['producer256_v3', 'producer256_v2', 'producer256', 'embed256', 'exit']
# A/B control: False disables Gumbel/Sequential-Halving selection (the one
# validated search win, +9.4%). Each setting gets its own run_name.
GUMBEL_SEARCH = True  #@param {type:"boolean"}

EXIT_FAMILY = ('producer256_v3', 'producer256_v2', 'producer256', 'embed256', 'exit')  # all run v2.exit_train
_base = {'producer256_v3': 'configs/v2_exit_producer256_v3.yaml',
         'producer256_v2': 'configs/v2_exit_producer256_v2.yaml',
         'producer256': 'configs/v2_exit_producer256.yaml',
         'embed256': 'configs/v2_exit_embed256.yaml',
         'exit': 'configs/v2_exit.yaml'}[PIPELINE]
cfg = load_v2_config(_base)
# producer256_v2 warm-starts from the iter-25 champion of the first re-anchor
# run (skips BC entirely — same embed-256 architecture). The others do fresh BC
# (producer256_v3 deliberately so: single-variable vs the champion recipe).
WARM_START = (f'{DRIVE_OUTPUT}/checkpoints/v2_exit_producer256_a100/ckpt_000025.pt'
              if PIPELINE == 'producer256_v2' else None)

# Device + Drive persistence
cfg.device = 'cuda'
cfg.save_dir = f'{DRIVE_OUTPUT}/checkpoints'
cfg.log_dir = f'{DRIVE_OUTPUT}/logs'

cfg.exit.gumbel_search = bool(GUMBEL_SEARCH)
_stem = {'producer256_v3': 'producer256_v3', 'producer256_v2': 'producer256_v2',
         'producer256': 'producer256', 'embed256': 'embed256', 'exit': 're128'}[PIPELINE]
cfg.run_name = f"v2_exit_{_stem}{'' if GUMBEL_SEARCH else '_noG'}_a100"

# Demo cache on Drive. producer256* use producer-MIRROR side-alternated demos
# (producer256_v2 only touches it on the no-warm-start fallback); seed the cache
# from a locally collected file to skip ~30-60 min of Colab collection:
#   rclone copy experiments/demos_producer_mirror_v1.pkl gdrive:orbit_wars_outputs/
_cache = {'producer256_v3': 'demos_producer_mirror_v1.pkl',
          'producer256_v2': 'demos_producer_mirror_v1.pkl',
          'producer256': 'demos_producer_mirror_v1.pkl',
          'embed256': 'demos_producer_embed256.pkl',
          'exit': 'demos_producer_embed256.pkl'}[PIPELINE]
cfg.imitation.bc_cache_path = f'{DRIVE_OUTPUT}/{_cache}'

# Colab horsepower overrides (the GPU is never the bottleneck -- collection and
# search are; fan everything across vCPUs and feed the A100 big batches).
cfg.imitation.bc_collect_workers = CPU_COUNT
cfg.imitation.bc_batch_size = 1024
cfg.exit.search_workers = CPU_COUNT
cfg.exit.collect_workers = CPU_COUNT
cfg.exit.collect_fast_env = True
cfg.exit.train_batch_size = 2048
cfg.eval.eval_workers = CPU_COUNT

print(f'PIPELINE      : {PIPELINE}')
print(f'run_name      : {cfg.run_name}')
print(f'config        : {_base}')
print(f'warm_start    : {WARM_START}')
print(f'model         : embed={cfg.model.embed_dim} layers={cfg.model.n_layers}')
print(f'BC            : enabled={cfg.imitation.enabled} expert={cfg.imitation.bc_expert} '
      f'vs {cfg.imitation.bc_demo_opponent} games={cfg.imitation.bc_games} '
      f'side_alt={cfg.imitation.bc_side_alternate} workers={cfg.imitation.bc_collect_workers}')
print(f'              : cache={cfg.imitation.bc_cache_path}')
print(f'ExIt          : iters={cfg.exit.iterations} games/iter={cfg.exit.games_per_iter} '
      f'opponent={cfg.exit.opponent} side_alt={cfg.exit.collect_side_alternate}')
print(f'              : gumbel={cfg.exit.gumbel_search} (sims={cfg.exit.gumbel_sims} '
      f'top_m={cfg.exit.gumbel_top_m}) collect_workers={cfg.exit.collect_workers}')
print(f'              : arrival_horizon={cfg.exit.arrival_horizon} '
      f'(margin={cfg.exit.arrival_settle_margin} cap={cfg.exit.arrival_horizon_cap})')
print(f'eval          : games={cfg.eval.eval_games} vs {cfg.eval.eval_opponents} '
      f'seed={cfg.eval.eval_seed} workers={cfg.eval.eval_workers} (side-alternated, paired)')
print('\nFINAL GATE = scripts/arena.py locally (pool + mirror vs producer/v5), not eval alone.')

In [ ]:
#@title 4. Train

import os, yaml
from v2.config import v2_config_to_dict

temp_cfg = '/tmp/v2_train_cfg.yaml'
with open(temp_cfg, 'w') as f:
    yaml.dump(v2_config_to_dict(cfg), f, sort_keys=True)
print(f'Wrote config -> {temp_cfg}')

CHECKPOINT_DIR = f'{cfg.save_dir}/{cfg.run_name}'

# Resume: continue from this run's ckpt_last.pt on Drive. The train script then
# SKIPS BC pretrain/demo collection, restores model+optimizer, advances the RNG
# seed past completed updates, and reloads cached demos if the imitation coef is
# still > 0 at the resume point.
RESUME = True  #@param {type:"boolean"}
resume_ckpt = f'{CHECKPOINT_DIR}/ckpt_last.pt'
resume_flag = ''
if RESUME and os.path.exists(resume_ckpt):
    resume_flag = f'--resume "{resume_ckpt}"'           # continue this run
    print(f'Resuming from {resume_ckpt}')
elif WARM_START and os.path.exists(WARM_START):
    resume_flag = f'--resume "{WARM_START}"'            # Phase 5 warm-start (skips BC)
    print(f'Warm-starting from {WARM_START}')
elif RESUME:
    print(f'RESUME=True but no checkpoint at {resume_ckpt}; starting fresh.')

module = 'v2.exit_train' if PIPELINE in EXIT_FAMILY else 'v2.train'
print(f'Launching: python -m {module} --config {temp_cfg} {resume_flag}\n')
# Drop kaggle_environments' OpenSpiel/core_harness INFO burst (emitted once per
# process at fast_env import -- main + every pool worker) from the live stream;
# real errors go to stderr (2>&1) and never match the filter. --line-buffered so
# training progress still streams live.
!python -m {module} --config {temp_cfg} {resume_flag} 2>&1 | grep -vE --line-buffered "^\[kaggle_environments\.(envs\.open_spiel_env|core_harness)"

print(f'\nCheckpoints saved to: {CHECKPOINT_DIR}')

## Submission

Builds a **trained-model bundle** dir *and* packs it into the **`submission.tar.gz`**
you actually upload: `main.py` + `ckpt_last.pt` + `submission_config.yaml` + the
`v2`/`src` packages. At match time Kaggle extracts the archive into
`/kaggle_simulations/agent/` and execs `main.py`, which auto-loads the checkpoint
and config from there.

⚠️ **Submit the `.tar.gz`, not the bundle directory, and `main.py`/`v2`/`src` must
sit at the *archive root*** (siblings of each other). If `v2/` ends up nested under
a bundle-dir prefix — or omitted entirely (e.g. uploading only `main.py`) — the
agent crashes with `ModuleNotFoundError: No module named 'v2'`. The cell below
builds the archive with `arcname` at root and self-checks it by extracting +
running one step before you upload.

- **v3** uses `v2/agent_v3.py` (the REAL feature pipeline), so the submitted
  policy sees byte-identical pair/comet features to training. The bundled
  `submission_config.yaml` tells it which feature flags + model dims to build.
- **ppo/exit** use the lighter inlined `v2/agent.py`.
- An **apex** baseline is also saved as a reliable fallback.

In [ ]:
#@title 5. Generate Submission

import shutil, yaml, tarfile, os, sys, tempfile
from pathlib import Path
from v2.config import v2_config_to_dict

sub_dir = Path(DRIVE_OUTPUT) / 'submissions'
sub_dir.mkdir(parents=True, exist_ok=True)

# Trained-model bundle
ckpt_src = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
bundle = sub_dir / f'{cfg.run_name}_bundle'
tarball = sub_dir / f'{cfg.run_name}_submission.tar.gz'
if ckpt_src.exists():
    bundle.mkdir(parents=True, exist_ok=True)
    # agent_v3 loads the REAL encode/decode + the bundled ckpt/config (no inlining drift).
    agent_src = 'v2/agent_v3.py'
    shutil.copy2(agent_src, bundle / 'main.py')
    shutil.copy2(ckpt_src, bundle / 'ckpt_last.pt')
    shutil.copytree('v2', bundle / 'v2', dirs_exist_ok=True)
    shutil.copytree('src', bundle / 'src', dirs_exist_ok=True)
    # Bundle the exact config so the agent rebuilds matching features/model.
    with open(bundle / 'submission_config.yaml', 'w') as f:
        yaml.dump(v2_config_to_dict(cfg), f, sort_keys=True)
    print(f'{PIPELINE} bundle: {bundle}')
    print('  contents: main.py + ckpt_last.pt + submission_config.yaml + v2/ + src/')

    # --- Build the actual submission.tar.gz ---------------------------------
    # CRITICAL: main.py + v2/ + src/ must sit at the ARCHIVE ROOT (arcname=name),
    # NOT nested under a bundle-dir prefix. Kaggle extracts the archive into
    # /kaggle_simulations/agent/, then execs /kaggle_simulations/agent/main.py.
    # If v2/ isn't a sibling of main.py there, main.py raises
    #   ModuleNotFoundError: No module named 'v2'
    ROOT_ENTRIES = ['main.py', 'ckpt_last.pt', 'submission_config.yaml', 'v2', 'src']
    if tarball.exists():
        tarball.unlink()
    with tarfile.open(tarball, 'w:gz') as tf:
        for name in ROOT_ENTRIES:
            tf.add(bundle / name, arcname=name)  # arcname=name -> top of archive

    # --- Self-check: FAITHFULLY replicate Kaggle's agent loader -------------
    # kaggle_environments.agent.get_last_callable does, in order:
    #   sys.path.append(dirname(main.py)); exec(code, {}); sys.path.pop()
    # then calls the returned agent LATER (lazily, first step). A naive
    # importlib.exec_module check would NOT catch a regression here because it
    # sets __file__ and never pops the path. We mimic the real sequence so a
    # broken bootstrap (a bundle import deferred until AFTER the pop) fails
    # HERE, locally -- not silently in the competition.
    with tempfile.TemporaryDirectory() as td:
        with tarfile.open(tarball) as tf:
            tops = sorted({m.name.split('/')[0] for m in tf.getmembers()})
            tf.extractall(td)
        assert tops == sorted(ROOT_ENTRIES), f'archive root wrong: {tops}'
        assert (Path(td) / 'v2' / '__init__.py').exists(), 'v2 not a package at root'
        main_path = str(Path(td) / 'main.py')
        cwd0 = os.getcwd()
        # Cold import: drop any cached bundle modules so resolution is real.
        for _m in [k for k in list(sys.modules)
                   if k in ('v2', 'src') or k.startswith(('v2.', 'src.'))]:
            del sys.modules[_m]
        try:
            os.chdir(td)                       # Kaggle: agent dir is cwd
            env = {}                           # Kaggle: bare namespace, NO __file__
            code = compile(open(main_path).read(), main_path, 'exec')
            sys.path.append(td)                # Kaggle: append exec_dir
            exec(code, env)                    # bundle imports must resolve HERE
            sys.path.pop()                     # Kaggle: pop right after exec
            agent_fn = [v for v in env.values() if callable(v)][-1]
            assert getattr(agent_fn, '__name__', None) == 'agent', 'agent must be last callable'
            from kaggle_environments import make
            obs = make('orbit_wars').reset(num_agents=2)[0]['observation']
            mv = agent_fn(obs)                 # lazy first call -- used to crash post-pop
        finally:
            os.chdir(cwd0)
    sz = tarball.stat().st_size / 1e6
    print(f'\nSUBMISSION TARBALL OK -> {tarball} ({sz:.1f} MB)')
    print(f'  archive root: {tops}')
    print(f'  self-check (Kaggle exec->pop semantics): agent() ran -> {len(mv)} move(s)')
else:
    print(f'No checkpoint at {ckpt_src} yet.')

print('\nUpload:')
if ckpt_src.exists():
    print(f'  kaggle competitions submit orbit-wars -f "{tarball}" -m "{cfg.run_name}"')
    print('  (submit the .tar.gz, NOT the bundle dir -- files must be at archive root)')


## Evaluation

Run the trained OrbitNet head-to-head vs apex and random.

In [ ]:
#@title 6. Evaluate Trained Model (coarse check -- the real gate is the local arena)

import torch
from pathlib import Path
from v2.model import OrbitNet
from v2.train import make_v2_eval_agent
from evaluation.evaluate import run_games, print_results
from agents import load_named_agent
from kaggle_environments.envs.orbit_wars.orbit_wars import random_agent

ckpt_path = Path(CHECKPOINT_DIR) / 'ckpt_last.pt'
if not ckpt_path.exists():
    print(f'No checkpoint at {ckpt_path}. Available:')
    for p in sorted(Path(DRIVE_OUTPUT, 'checkpoints').rglob('ckpt_last.pt')):
        print(f'  {p}')
else:
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = OrbitNet(cfg.model).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model'])
    model.eval()
    print(f'Loaded {ckpt_path} (iter/update {ckpt.get("update", "?")})')
    eval_agent = make_v2_eval_agent(model, cfg, device)

    N = 20  # coarse: P0-only, unpaired. Trust run_periodic_eval / eval_fast / arena.
    print(f'\nRL vs producer ({N} games):')
    print_results('v2', 'producer', run_games(eval_agent, load_named_agent('producer'), n_games=N))
    print(f'RL vs Random ({N} games):')
    print_results('v2', 'random', run_games(eval_agent, random_agent, n_games=N))

## Monitoring

Each run writes a resume-safe **`eval_history.csv`** to its Drive log dir
(appended + closed every eval, so the FUSE mount actually syncs it and the
full curve survives Colab session restarts). Plot it any time — even from a
second runtime while training continues — with the eval-curve cell below.
TensorBoard (last cell) shows train/*, eval/*, bc/* and can block cells below it.

In [ ]:
#@title 7. Plot eval win-rate curve (from eval_history.csv)

from pathlib import Path
from IPython.display import Image, display

csv_path = Path(CHECKPOINT_DIR.replace('/checkpoints/', '/logs/')) / 'eval_history.csv'
out_png = f'{DRIVE_OUTPUT}/logs/{cfg.run_name}_eval.png'
if csv_path.exists():
    !python scripts/plot_eval_curve.py --csv "{csv_path}" --out "{out_png}" --title "{cfg.run_name} — eval win rate"
    display(Image(out_png))
else:
    print(f'No eval_history.csv yet at {csv_path} (appears after the first eval).')

In [ ]:
#@title 8. TensorBoard

%load_ext tensorboard
%tensorboard --logdir {DRIVE_OUTPUT}/logs